In [0]:
from pyspark.sql.functions import col,lit,max,sum
import uuid
from datetime import datetime

In [0]:
spark.sql("""
          create table if not exists novacart.bronze.ingestioncontrol(
              layer string,
              table string,
              ts_col string,
              pk_col string,
              last_successful_ts timestamp,
              last_successful_pk bigint,
              last_run_id string,
              rows_written bigint,
              row_status string,
              updated_at timestamp
          )
          """)

In [0]:
tables_config = {
    "orders":{"pk_col" : "order_id","ts_col" : "updated_at"},
    "payments":{"pk_col" : "payment_id","ts_col" : "processed_at"},
    "products":{"pk_col" : "product_id","ts_col" : "updated_at"}
}
bronze_run_id = str(uuid.uuid4())
print(bronze_run_id)

In [0]:
def get_last_successfil_watermark(table_name:str):
    cntrl = (
        spark.table("novacart.bronze.ingestioncontrol")
             .filter((col("layer") == "bronze") & (col("table") == table_name) & (col("row_status") == "Success"))
             .orderBy(col("updated_at").desc())
             .limit(1)
    )
    rows = cntrl.collect()
    if not rows:
        return None,None
    
    return rows[0].last_successful_ts,rows[0].last_successful_pk

